
# Ejercicio de Persecusión


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Persecución Automóvil vs Policía</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Observa cómo avanza el tiempo.</p>
    
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #f73bc5; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir / Pausar</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer;">⏹ Reiniciar</button>
            <div style="font-size: 1.1em; font-weight: bold; margin-left: 10px;">
                Tiempo: <span id="time_display_UID" style="color: #dc3545;">0.0</span> s
            </div>
        </div>
        
        <!-- Control de velocidad de la simulación -->
        <div style="display: flex; align-items: center; gap: 10px; background: #e9ecef; padding: 8px 15px; border-radius: 5px;">
            <label style="font-weight: 600; font-size: 0.9em; margin: 0;">Velocidad de Simulación:</label>
            <input type="range" id="speed_UID" min="0.2" max="2" step="0.2" value="1.0" style="vertical-align: middle;">
            <span id="speed_val_UID" style="font-weight: bold; font-family: monospace; width: 35px;">1.0x</span>
        </div>
    </div>
    
    <div style="text-align: center; position: relative; overflow: hidden; display: flex; flex-direction: column; gap: 10px;">
        <!-- Canvas de la persecución física -->
        <canvas id="canvas_sim_UID" width="950" height="120" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
        <!-- Canvas de los gráficos -->
        <canvas id="canvas_graphs_UID" width="950" height="300" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="margin-top: 15px; font-size: 0.9em; display: flex; justify-content: center; gap: 30px;">
        <div><span style="display:inline-block; width:15px; height:15px; background:#0000FF; border-radius:3px; vertical-align:middle;"></span> Automóvil (v cte = 22 m/s)</div>
        <div><span style="display:inline-block; width:15px; height:15px; background:#FF0000; border-radius:3px; vertical-align:middle;"></span> Policía (Espera 4 s, acelera a 30 m/s)</div>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvasSim = document.getElementById('canvas_sim_UID');
            const canvasGraph = document.getElementById('canvas_graphs_UID');
            if (!canvasSim || !canvasGraph) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctxSim = canvasSim.getContext('2d');
            const ctxG = canvasGraph.getContext('2d');
            
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_UID');
            const time_display = document.getElementById('time_display_UID');
            const speed_slider = document.getElementById('speed_UID');
            const speed_val = document.getElementById('speed_val_UID');
            
            let t = 0;
            let isPlaying = false;
            let animationId;
            let lastTime = 0; 
            const tMax = 34; 
            
            // --- Variables de Audio Web API ---
            let audioCtx = null;
            let osc = null;
            let gainNode = null;
            let isSirenPlaying = false;
            
            function initAudio() {
                if (!audioCtx) {
                    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                }
                if (audioCtx.state === 'suspended') {
                    audioCtx.resume();
                }
            }

            function startSiren() {
                if (isSirenPlaying) return;
                initAudio();
                osc = audioCtx.createOscillator();
                gainNode = audioCtx.createGain();
                
                osc.type = 'square'; // Sonido estridente tipo alarma
                osc.connect(gainNode);
                gainNode.connect(audioCtx.destination);
                
                gainNode.gain.value = 0; // Inicia en silencio
                osc.start();
                isSirenPlaying = true;
            }

            function stopSiren() {
                if (osc) {
                    osc.stop();
                    osc.disconnect();
                    osc = null;
                }
                if (gainNode) {
                    gainNode.disconnect();
                    gainNode = null;
                }
                isSirenPlaying = false;
            }

            function updateSiren(currentTime_s, xAuto, xPolice) {
                // Solo encender sirena si está en play y pasó el tiempo de espera (t=4)
                if (isPlaying && currentTime_s >= 4 && currentTime_s < tMax) {
                    if (!isSirenPlaying) startSiren();
                    
                    // Calcular distancia (máximo en t=4 es 88m, disminuye hasta 0)
                    let distance = Math.max(0, xAuto - xPolice);
                    
                    // Volumen: aumenta a medida que la distancia disminuye (máx vol 0.15 para no ensordecer)
                    let maxDist = 88;
                    let vol = 0.02 + 0.13 * (1 - Math.min(distance, maxDist) / maxDist);
                    
                    // Frecuencia alternante tipo sirena europea (Wee-Woo)
                    let pitch = (Math.floor(Date.now() / 350) % 2 === 0) ? 800 : 600;
                    
                    osc.frequency.setTargetAtTime(pitch, audioCtx.currentTime, 0.05);
                    gainNode.gain.setTargetAtTime(vol, audioCtx.currentTime, 0.1);
                } else {
                    stopSiren();
                }
            }
            // ----------------------------------

            speed_slider.addEventListener('input', () => {
                speed_val.innerText = parseFloat(speed_slider.value).toFixed(1) + 'x';
            });
            
            // Funciones físicas
            function getAutoState(time) {
                return { x: 22 * time, v: 22, a: 0 };
            }
            
            function getPoliceState(time) {
                if (time <= 4) return { x: 0, v: 0, a: 0 };
                if (time <= 14) {
                    let a = 3.0; 
                    let dt_p = time - 4;
                    return { x: 0.5 * a * dt_p * dt_p, v: a * dt_p, a: a };
                }
                let d_accel = 0.5 * 3.0 * Math.pow(10, 2); 
                let dt_const = time - 14;
                return { x: d_accel + 30 * dt_const, v: 30, a: 0 };
            }

            // --- Funciones para dibujar vehículos ---
            function drawModernCar(ctx, x, y) {
                ctx.save();
                ctx.translate(x, y);
                
                // Sombra
                ctx.fillStyle = "rgba(0,0,0,0.3)";
                ctx.beginPath(); ctx.ellipse(20, 6, 22, 4, 0, 0, Math.PI*2); ctx.fill();

                // Carrocería con gradiente moderno y aerodinámico
                let grad = ctx.createLinearGradient(0, -10, 40, -10);
                grad.addColorStop(0, '#ff0055'); // Magenta trasero
                grad.addColorStop(0.5, '#ff4400');
                grad.addColorStop(1, '#ffaa00'); // Naranja delantero
                
                ctx.fillStyle = grad;
                ctx.beginPath();
                ctx.moveTo(0, -2);
                ctx.lineTo(3, -10);  // maletero
                ctx.lineTo(15, -14); // techo trasero
                ctx.lineTo(25, -14); // techo delantero
                ctx.lineTo(38, -6);  // capó inclinado
                ctx.lineTo(42, -2);  // parachoques delantero
                ctx.lineTo(42, 4);
                ctx.lineTo(0, 4);
                ctx.fill();
                
                // Ventanas oscuras
                ctx.fillStyle = '#222';
                ctx.beginPath();
                ctx.moveTo(7, -9);
                ctx.lineTo(15, -12);
                ctx.lineTo(23, -12);
                ctx.lineTo(32, -6);
                ctx.lineTo(24, -6);
                ctx.lineTo(14, -6);
                ctx.fill();
                
                // Detalles (Luces)
                ctx.fillStyle = '#ff0000'; // luz trasera
                ctx.fillRect(0, -2, 2, 4);
                ctx.fillStyle = '#ffffff'; // luz delantera
                ctx.beginPath(); ctx.moveTo(40, -2); ctx.lineTo(42, -2); ctx.lineTo(42, 2); ctx.fill();

                // Ruedas deportivas
                ctx.fillStyle = '#111';
                ctx.beginPath(); ctx.arc(8, 4, 5, 0, Math.PI*2); ctx.fill();
                ctx.beginPath(); ctx.arc(33, 4, 5, 0, Math.PI*2); ctx.fill();
                
                // Rines
                ctx.fillStyle = '#dddddd';
                ctx.beginPath(); ctx.arc(8, 4, 2.5, 0, Math.PI*2); ctx.fill();
                ctx.beginPath(); ctx.arc(33, 4, 2.5, 0, Math.PI*2); ctx.fill();

                ctx.restore();
            }

            function drawPoliceMoto(ctx, x, y, currentTime) {
                ctx.save();
                ctx.translate(x, y);
                
                // Sombra
                ctx.fillStyle = "rgba(0,0,0,0.3)";
                ctx.beginPath(); ctx.ellipse(12, 6, 12, 3, 0, 0, Math.PI*2); ctx.fill();

                // Ruedas
                ctx.fillStyle = '#111';
                ctx.beginPath(); ctx.arc(4, 4, 4.5, 0, Math.PI*2); ctx.fill();
                ctx.beginPath(); ctx.arc(20, 4, 4.5, 0, Math.PI*2); ctx.fill();
                
                // Rines
                ctx.fillStyle = '#fff';
                ctx.beginPath(); ctx.arc(4, 4, 2, 0, Math.PI*2); ctx.fill();
                ctx.beginPath(); ctx.arc(20, 4, 2, 0, Math.PI*2); ctx.fill();
                
                // Chasis principal
                ctx.fillStyle = '#ffffff';
                ctx.beginPath();
                ctx.moveTo(2, 0);
                ctx.lineTo(8, -8);
                ctx.lineTo(16, -6);
                ctx.lineTo(22, 0);
                ctx.fill();
                
                // Detalles azules de la policía
                ctx.fillStyle = '#0d6efd';
                ctx.beginPath(); ctx.moveTo(8, -8); ctx.lineTo(14, -6); ctx.lineTo(14, -2); ctx.lineTo(6, -2); ctx.fill();

                // Parabrisas
                ctx.strokeStyle = 'rgba(173, 216, 230, 0.8)';
                ctx.lineWidth = 2.5;
                ctx.beginPath(); ctx.moveTo(16, -6); ctx.lineTo(19, -13); ctx.stroke();
                
                // Baliza de Sirena parpadeante
                let isRed = Math.floor(currentTime * 6) % 2 === 0; // Parpadeo basado en el tiempo
                ctx.fillStyle = isRed ? '#dc3545' : '#0d6efd';
                ctx.beginPath(); ctx.arc(2, -9, 2.5, 0, Math.PI*2); ctx.fill();
                // Destello luminoso
                if (isPlaying && currentTime >= 4) {
                    ctx.fillStyle = isRed ? 'rgba(220, 53, 69, 0.4)' : 'rgba(13, 110, 253, 0.4)';
                    ctx.beginPath(); ctx.arc(2, -9, 6, 0, Math.PI*2); ctx.fill();
                }

                // Policía (Piloto)
                ctx.fillStyle = '#111'; // Traje negro
                ctx.beginPath(); ctx.arc(10, -14, 3, 0, Math.PI*2); ctx.fill(); // Casco
                // Brazo y torso
                ctx.strokeStyle = '#222';
                ctx.lineWidth = 3;
                ctx.lineCap = 'round';
                ctx.beginPath(); 
                ctx.moveTo(8, -8); // Asiento
                ctx.lineTo(10, -12); // Hombro
                ctx.lineTo(15, -9); // Manubrio
                ctx.stroke();

                ctx.restore();
            }
            // ----------------------------------------

            function drawSim() {
                ctxSim.clearRect(0, 0, canvasSim.width, canvasSim.height);
                
                // Dibujar calle
                ctxSim.fillStyle = "#e9ecef";
                ctxSim.fillRect(0, 30, canvasSim.width, 70); 
                ctxSim.strokeStyle = "#adb5bd";
                ctxSim.lineWidth = 2;
                ctxSim.beginPath();
                ctxSim.setLineDash([20, 20]);
                ctxSim.moveTo(0, 65); 
                ctxSim.lineTo(canvasSim.width, 65);
                ctxSim.stroke();
                ctxSim.setLineDash([]);
                
                let sA = getAutoState(t);
                let sP = getPoliceState(t);
                
                let scaleX = canvasSim.width / 850; 
                
                // Dibujar Vehículos
                drawModernCar(ctxSim, sA.x * scaleX, 48); 
                drawPoliceMoto(ctxSim, sP.x * scaleX, 85, t); 
                
                // Marcador de meta (748m)
                ctxSim.fillStyle = "#198754";
                ctxSim.fillRect(748 * scaleX, 25, 6, 80);
                
                ctxSim.fillStyle = "#111";
                ctxSim.font = "bold 13px Arial";
                ctxSim.fillText("748 m", 748 * scaleX - 15, 18);

                // Llamar a la actualización de audio
                updateSiren(t, sA.x, sP.x);
            }

            function drawGraphs() {
                ctxG.clearRect(0, 0, canvasGraph.width, canvasGraph.height);
                
                let margin = 40;
                let w1 = 230; 
                let h1 = 200; 
                
                let offsetX = 50;  
                let offsetV = 360; 
                let offsetA = 670; 
                
                // --- Ejes ---
                ctxG.strokeStyle = "#000";
                
                // Posición
                ctxG.beginPath();
                ctxG.moveTo(offsetX, margin); ctxG.lineTo(offsetX, margin + h1);
                ctxG.lineTo(offsetX + w1, margin + h1);
                ctxG.stroke();
                ctxG.fillStyle = "#000";
                ctxG.fillText("Posición (m)", offsetX - 20, margin - 10);
                ctxG.fillText("Tiempo (s)", offsetX + w1 - 30, margin + h1 + 20);
                
                // Velocidad
                ctxG.beginPath();
                ctxG.moveTo(offsetV, margin); ctxG.lineTo(offsetV, margin + h1); 
                ctxG.lineTo(offsetV + w1, margin + h1); 
                ctxG.stroke();
                ctxG.fillText("Velocidad (m/s)", offsetV - 20, margin - 10);
                ctxG.fillText("Tiempo (s)", offsetV + w1 - 30, margin + h1 + 20);

                // Aceleración
                ctxG.beginPath();
                ctxG.moveTo(offsetA, margin); ctxG.lineTo(offsetA, margin + h1); 
                ctxG.lineTo(offsetA + w1, margin + h1); 
                ctxG.stroke();
                ctxG.fillText("Aceleración (m/s²)", offsetA - 20, margin - 10);
                ctxG.fillText("Tiempo (s)", offsetA + w1 - 30, margin + h1 + 20);
                
                let scaleT = w1 / 40; 
                let scalePos = h1 / 850; 
                let scaleVel = h1 / 40;  
                let scaleAcc = h1 / 5;   
                
                ctxG.lineWidth = 2.5;
                
                // --- Posición ---
                ctxG.beginPath(); ctxG.strokeStyle = "#0000FF"; // Color Auto (Azul)
                for(let i=0; i<=t; i+=0.5) {
                    let val = getAutoState(i);
                    let px = offsetX + i * scaleT;
                    let py = margin + h1 - val.x * scalePos;
                    i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                } ctxG.stroke();
                
                ctxG.beginPath(); ctxG.strokeStyle = "#FF0000"; // Color Moto (Rojo)
                for(let i=0; i<=t; i+=0.5) {
                    let val = getPoliceState(i);
                    let px = offsetX + i * scaleT;
                    let py = margin + h1 - val.x * scalePos;
                    i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                } ctxG.stroke();

                // --- Velocidad ---
                ctxG.beginPath(); ctxG.strokeStyle = "#0000FF"; 
                for(let i=0; i<=t; i+=0.5) {
                    let val = getAutoState(i);
                    let px = offsetV + i * scaleT;
                    let py = margin + h1 - val.v * scaleVel;
                    i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                } ctxG.stroke();

                ctxG.beginPath(); ctxG.strokeStyle = "#FF0000"; 
                for(let i=0; i<=t; i+=0.5) {
                    let val = getPoliceState(i);
                    let px = offsetV + i * scaleT;
                    let py = margin + h1 - val.v * scaleVel;
                    i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                } ctxG.stroke();

                // --- Aceleración ---
                ctxG.beginPath(); ctxG.strokeStyle = "#0000FF"; 
                for(let i=0; i<=t; i+=0.1) {
                    let val = getAutoState(i);
                    let px = offsetA + i * scaleT;
                    let py = margin + h1 - val.a * scaleAcc;
                    i===0 ? ctxG.moveTo(px, py - 1) : ctxG.lineTo(px, py - 1);
                } ctxG.stroke();

                ctxG.beginPath(); ctxG.strokeStyle = "#FF0000"; 
                for(let i=0; i<=t; i+=0.1) {
                    let val = getPoliceState(i);
                    let px = offsetA + i * scaleT;
                    let py = margin + h1 - val.a * scaleAcc;
                    i===0 ? ctxG.moveTo(px, py) : ctxG.lineTo(px, py);
                } ctxG.stroke();
                
                // Marcar punto de encuentro
                if (t >= 34) {
                    let pxMeet = offsetX + 34 * scaleT;
                    let pyMeet = margin + h1 - 748 * scalePos;
                    ctxG.fillStyle = "#dc3545"; 
                    ctxG.beginPath(); ctxG.arc(pxMeet, pyMeet, 5, 0, Math.PI*2); ctxG.fill();
                    ctxG.fillText("¡Encuentro!", pxMeet - 25, pyMeet - 15);
                }
            }

            function animate(timestamp) {
                if (!isPlaying) return;
                
                if (!lastTime) lastTime = timestamp;
                
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                let speed_multiplier = parseFloat(speed_slider.value);
                
                t += deltaTime * speed_multiplier;
                
                if (t >= tMax) {
                    t = tMax;
                    isPlaying = false;
                    stopSiren(); // Detener audio al terminar
                }
                
                time_display.innerText = t.toFixed(1);
                drawSim();
                drawGraphs();
                
                if(isPlaying) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            btn_play.addEventListener('click', () => {
                initAudio(); // Requisito de navegadores: inicializar contexto de audio con un click
                
                isPlaying = !isPlaying;
                if (t >= tMax) {
                    t = 0; 
                }
                
                if (isPlaying) {
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                } else {
                    stopSiren();
                }
            });

            btn_reset.addEventListener('click', () => {
                isPlaying = false;
                t = 0;
                lastTime = 0;
                time_display.innerText = "0.0";
                speed_slider.value = 1.0; 
                speed_val.innerText = "1.0x";
                stopSiren();
                drawSim();
                drawGraphs();
            });
            
            drawSim();
            drawGraphs();
        }
        
        initSimulation();
    })();
</script>
""".replace('UID', uid)

display(HTML(simulacion_html))

In [2]:
from IPython.display import display, HTML
import random

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

quiz_cinematica_html = """
<style>
    .fisi-quiz-container { border: 1px solid #e0e0e0; padding: 15px; margin-bottom: 25px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; }
    .fisi-question { font-weight: 600; margin-bottom: 15px; color: #2c3e50; }
    .fisi-option { margin-bottom: 10px; display: flex; align-items: flex-start; cursor: pointer; }
    .fisi-option input { margin-top: 4px; margin-right: 10px; }
    .fisi-check-btn { background-color: #f73bc5; color: white; border: none; padding: 8px 20px; border-radius: 4px; cursor: pointer; font-size: 14px; margin-top: 10px; font-weight: 500; }
    .fisi-check-btn:hover { background-color: #ab2988; }
    .fisi-feedback { margin-top: 15px; padding: 12px; border-radius: 6px; display: none; font-size: 0.95em; }
</style>

<!-- Pregunta 1: Análisis visual básico -->
<div class="fisi-quiz-container" id="quiz-cin-1">
    <div class="fisi-question">1. De acuerdo a la simulación y a los gráficos generados, ¿qué tipo de movimiento describe el automóvil desde t = 0 s y cómo se representa en el gráfico de velocidad vs. tiempo?</div>
    <label class="fisi-option"><input type="radio" name="qcin1" value="a" data-fb="Incorrecto. Una recta diagonal indicaría que la velocidad está cambiando (aceleración)."> <span>Movimiento con aceleración constante, representado por una línea recta diagonal creciente.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin1" value="b" data-correct="true" data-fb="¡Correcto! El automóvil viaja a una rapidez constante de 22.0 m/s, por lo que su velocidad no cambia, trazando una línea completamente horizontal a lo largo del tiempo."> <span>Movimiento Rectilíneo Uniforme (MRU), representado por una línea recta horizontal.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin1" value="c" data-fb="Incorrecto. El automóvil no parte del reposo; tiene velocidad desde el instante inicial."> <span>Movimiento desde el reposo, representado por una curva parabólica.</span></label>
    <button class="fisi-check-btn" id="btn-check-cin1">Comprobar</button><div class="fisi-feedback" id="feedback-cin1"></div>
</div>

<!-- Pregunta 2: Extracción de información temporal -->
<div class="fisi-quiz-container" id="quiz-cin-2">
    <div class="fisi-question">2. Observando el comportamiento de la motocicleta de policía, ¿qué ocurre físicamente durante los primeros 4 segundos de la simulación?</div>
    <label class="fisi-option"><input type="radio" name="qcin2" value="a" data-fb="Incorrecto. Durante este tiempo el gráfico de velocidad de la policía se mantiene en cero."> <span>Se mueve a la misma velocidad constante que el automóvil.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin2" value="b" data-fb="Incorrecto. La aceleración ocurre después de este período de tiempo."> <span>Comienza a acelerar inmediatamente desde t = 0 s.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin2" value="c" data-correct="true" data-fb="¡Correcto! El policía parte del reposo 4 s después de que el automóvil pasa frente a él. En el gráfico de posición y velocidad, sus valores son cero durante este tramo."> <span>Permanece en reposo en la posición inicial (x = 0 m).</span></label>
    <button class="fisi-check-btn" id="btn-check-cin2">Comprobar</button><div class="fisi-feedback" id="feedback-cin2"></div>
</div>

<!-- Pregunta 3: Interpretación geométrica (Áreas) -->
<div class="fisi-quiz-container" id="quiz-cin-3">
    <div class="fisi-question">3. ¿Qué representa físicamente el área bajo la curva en el gráfico de velocidad vs. tiempo para ambos vehículos en el instante en que se encuentran (t = 34 s)?</div>
    <label class="fisi-option"><input type="radio" name="qcin3" value="a" data-correct="true" data-fb="¡Excelente! El área bajo la curva en un gráfico de velocidad-tiempo representa la distancia recorrida o el desplazamiento. Como se encuentran a los 34 s, ambas áreas deben ser iguales y corresponden a 748 m."> <span>La distancia total recorrida por cada vehículo desde el inicio.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin3" value="b" data-fb="Incorrecto. La aceleración se asocia a la pendiente de la curva de velocidad, no al área bajo ella."> <span>La aceleración instantánea de los vehículos en ese segundo.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin3" value="c" data-fb="Incorrecto. La diferencia de velocidades se lee directamente observando los valores en el eje Y del gráfico de velocidad."> <span>La diferencia de velocidades entre el automóvil y la policía.</span></label>
    <button class="fisi-check-btn" id="btn-check-cin3">Comprobar</button><div class="fisi-feedback" id="feedback-cin3"></div>
</div>

<!-- Pregunta 4: Cálculo de aceleración -->
<div class="fisi-quiz-container" id="quiz-cin-4">
    <div class="fisi-question">4. La motocicleta de policía inicia su persecución en t = 4 s y alcanza una velocidad constante de 30.0 m/s tras acelerar por 10 s. ¿Cuál es la magnitud de su aceleración durante ese intervalo?</div>
    <label class="fisi-option"><input type="radio" name="qcin4" value="a" data-fb="Incorrecto. Recuerda que la aceleración se calcula como <span>a = &Delta;v / &Delta;t</span>. Revisa tus cálculos."> <span>7.5 m/s<sup>2</span></label>
    <label class="fisi-option"><input type="radio" name="qcin4" value="b" data-fb="Incorrecto. El intervalo de tiempo de aceleración es de 10 s, no de 14 s."> <span>2.1 m/s<sup>2</span></label>
    <label class="fisi-option"><input type="radio" name="qcin4" value="c" data-correct="true" data-fb="¡Correcto! Utilizando la pendiente del gráfico de velocidad vs. tiempo: a = (30 - 0) m/s / (14 - 4) s = 3.0 m/s<sup>2"> <span>3.0 m/s<sup>2</span></label>
    <button class="fisi-check-btn" id="btn-check-cin4">Comprobar</button><div class="fisi-feedback" id="feedback-cin4"></div>
</div>

<!-- Pregunta 5: Análisis avanzado (Intersección gráfica) -->
<div class="fisi-quiz-container" id="quiz-cin-5">
    <div class="fisi-question">5. En el análisis geométrico de estos movimientos, ¿cómo podemos identificar con precisión el instante exacto y la ubicación del encuentro utilizando el gráfico de Posición vs. Tiempo?</div>
    <label class="fisi-option"><input type="radio" name="qcin5" value="a" data-correct="true" data-fb="¡Perfecto! El encuentro ocurre en el instante en que ambos vehículos ocupan el mismo lugar en el espacio. Gráficamente, esto es la intersección de ambas curvas en el gráfico de Posición vs. Tiempo."> <span>Buscando el punto coordenado exacto donde ambas curvas se interceptan o cruzan.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin5" value="b" data-fb="Incorrecto. Cuando las pendientes son iguales, significa que ambos vehículos van a la misma velocidad, pero no necesariamente ocupan la misma posición."> <span>Identificando el punto donde ambas curvas tienen exactamente la misma pendiente.</span></label>
    <label class="fisi-option"><input type="radio" name="qcin5" value="c" data-fb="Incorrecto. Cuando la curva de velocidad de la policía cruza la del auto, significa que alcanzaron la misma rapidez (22 m/s), pero en ese momento el auto aún va metros por delante."> <span>Ubicando el momento en que la curva de la policía supera en altura a la curva del automóvil en el gráfico de velocidad.</span></label>
    <button class="fisi-check-btn" id="btn-check-cin5">Comprobar</button><div class="fisi-feedback" id="feedback-cin5"></div>
</div>

<script>
    (function() {
        function setupQuiz(btnId, radioName, feedbackId) {
            var btn = document.getElementById(btnId);
            if (!btn) return;
            btn.addEventListener('click', function() {
                var radios = document.getElementsByName(radioName);
                var selected = null;
                var feedbackDiv = document.getElementById(feedbackId);
                for (var i = 0; i < radios.length; i++) { if (radios[i].checked) { selected = radios[i]; break; } }
                feedbackDiv.style.display = 'block';
                if (!selected) {
                    feedbackDiv.style.backgroundColor = '#fff3cd'; feedbackDiv.style.color = '#856404'; feedbackDiv.style.border = '1px solid #ffeeba'; feedbackDiv.innerHTML = '⚠️ Por favor, selecciona una opción.'; return;
                }
                var isCorrect = selected.getAttribute('data-correct') === 'true';
                var msg = selected.getAttribute('data-fb');
                if (isCorrect) {
                    feedbackDiv.style.backgroundColor = '#d4edda'; feedbackDiv.style.color = '#155724'; feedbackDiv.style.border = '1px solid #c3e6cb'; feedbackDiv.innerHTML = '✅ <strong>¡Correcto!</strong><br>' + msg;
                } else {
                    feedbackDiv.style.backgroundColor = '#f8d7da'; feedbackDiv.style.color = '#721c24'; feedbackDiv.style.border = '1px solid #f5c6cb'; feedbackDiv.innerHTML = '❌ <strong>Incorrecto.</strong><br>' + msg;
                }
            });
        }
        setTimeout(function() { 
            setupQuiz('btn-check-cin1', 'qcin1', 'feedback-cin1'); 
            setupQuiz('btn-check-cin2', 'qcin2', 'feedback-cin2'); 
            setupQuiz('btn-check-cin3', 'qcin3', 'feedback-cin3'); 
            setupQuiz('btn-check-cin4', 'qcin4', 'feedback-cin4'); 
            setupQuiz('btn-check-cin5', 'qcin5', 'feedback-cin5'); 
        }, 500);
    })();
</script>
"""

display(HTML(quiz_cinematica_html))